# Load origin data

In [1]:
import pandas as pd
import numpy as np

In [2]:
import os

origin_data_dir = "data/origin_data"
transactions_file = "credit_card_transactions-ibm_v2.csv"
users_file = "sd254_users.csv"
cards_file = "sd254_cards.csv"

transactions_df = pd.read_csv(os.path.join(origin_data_dir, transactions_file))
transactions_df

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24386895,1999,1,2020,2,27,22:23,$-54.00,Chip Transaction,-5162038175624867091,Merrimack,NH,3054.0,5541,NaN,No
24386896,1999,1,2020,2,27,22:24,$54.00,Chip Transaction,-5162038175624867091,Merrimack,NH,3054.0,5541,NaN,No
24386897,1999,1,2020,2,28,07:43,$59.15,Chip Transaction,2500998799892805156,Merrimack,NH,3054.0,4121,NaN,No
24386898,1999,1,2020,2,28,20:10,$43.12,Chip Transaction,2500998799892805156,Merrimack,NH,3054.0,4121,NaN,No


In [3]:
users_df = pd.read_csv(os.path.join(origin_data_dir, users_file))
users_df

,Person,Current Age,Retirement Age,Birth Year,Birth Month,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5
1,Sasha Sadr,53,68,1966,12,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,$37891,$77254,$191349,701,5
2,Saanvi Lee,81,67,1938,11,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,$22681,$33483,$196,698,5
3,Everlee Clark,63,63,1957,1,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,$163145,$249925,$202328,722,4
4,Kyle Peterson,43,70,1976,9,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,$53797,$109687,$183855,675,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Jose Faraday,32,70,1987,7,Male,6577 Lexington Lane,9.0,Freeport,NY,11520,40.65,-73.58,$23550,$48010,$87837,703,3
1996,Ximena Richardson,62,65,1957,11,Female,2 Elm Drive,955.0,Independence,KY,41051,38.95,-84.54,$24218,$49378,$104480,740,4
1997,Annika Russell,47,67,1973,1,Female,276 Fifth Boulevard,NaN,Elizabeth,NJ,7201,40.66,-74.19,$15175,$30942,$71066,779,3
1998,Juelz Roman,66,60,1954,2,Male,259 Valley Boulevard,NaN,Camp Hill,PA,17011,40.24,-76.92,$25336,$54654,$27241,618,1


In [4]:
cards_df = pd.read_csv(os.path.join(origin_data_dir, cards_file))
cards_df

,User,CARD INDEX,Card Brand,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,0,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,0,1,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,0,2,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,0,3,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,0,4,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6141,1997,1,Amex,Credit,300609782832003,01/2024,663,YES,1,$6900,11/2000,2013,No
6142,1997,2,Visa,Credit,4718517475996018,01/2021,492,YES,2,$5700,04/2012,2012,No
6143,1998,0,Mastercard,Credit,5929512204765914,08/2020,237,NO,2,$9200,02/2012,2012,No
6144,1999,0,Mastercard,Debit,5589768928167462,01/2020,630,YES,1,$28074,01/2020,2020,No


# Transactions data preprocessing - 1

#### "Mercaht City" 열이 "ONLINE"인데 "USE Chip"열이 "Chip Transaction"인 경우 "Online Transaction"으로 변경

In [5]:
transactions_df[(transactions_df['Merchant City'] == "ONLINE")]['Use Chip'].unique()

array(['Online Transaction', 'Chip Transaction'], dtype=object)

In [6]:
transactions_df.loc[(transactions_df["Merchant City"] == "ONLINE") & (transactions_df["Use Chip"] == "Chip Transaction"), "Use Chip"] = "Online Transaction"

In [7]:
transactions_df[(transactions_df['Merchant City'] == "ONLINE")]['Use Chip'].unique()

array(['Online Transaction'], dtype=object)

#### Year, Month, Day, Time 열을 합쳐서 하나의 DateTime으로 변경 후 정렬

In [8]:
transactions_df['Datetime'] = pd.to_datetime(
    transactions_df['Year'].astype(str) + '-' + 
    transactions_df['Month'].astype(str) + '-' + 
    transactions_df['Day'].astype(str) + ' ' + 
    transactions_df['Time']
)

In [9]:
transactions_df = transactions_df.drop(columns=["Year", "Month", "Day", "Time"])
cols = ["Datetime"] + [c for c in transactions_df.columns if c != "Datetime"]
transactions_df = transactions_df[cols].sort_values(by="Datetime")
transactions_df

,Datetime,User,Card,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
9196264,1991-01-02 07:10:00,791,1,$68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,No
9196265,1991-01-02 07:17:00,791,1,$-68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,No
9196266,1991-01-02 07:21:00,791,1,$113.62,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,No
9196267,1991-01-02 17:30:00,791,1,$114.73,Swipe Transaction,-7269691894846892021,Burke,VA,22015.0,5411,NaN,No
9196268,1991-01-03 09:03:00,791,1,$251.71,Swipe Transaction,-3693650930986299431,Burke,VA,22015.0,4814,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...,...
20357329,2020-02-28 23:51:00,1659,2,$7.67,Chip Transaction,7231700044622779845,Cosby,TN,37722.0,5300,NaN,No
10296971,2020-02-28 23:53:00,863,1,$49.06,Chip Transaction,7654254764356253071,North Brunswick,NJ,8902.0,5912,NaN,No
16828012,2020-02-28 23:56:00,1366,2,$132.73,Chip Transaction,-7398558035733466800,Rockville Centre,NY,11570.0,5812,NaN,No
15999580,2020-02-28 23:56:00,1300,0,$51.29,Online Transaction,-6458444334611773637,ONLINE,NaN,NaN,4784,NaN,No


#### "Errors?"의 열 이름을 "Errors"로 변경 후 Error가 여러개일 경우 list로 변환

In [10]:
transactions_df.rename(columns={'Errors?': 'Errors'}, inplace=True)

In [11]:
transactions_df['Errors'] = transactions_df['Errors'].str.split(',')

In [12]:
for trans in transactions_df.iloc:
    if trans['Errors'] is not np.nan:
        print(trans['Errors'])
        print(trans)
        break

['Insufficient Balance']
Datetime             1991-03-14 18:24:00
User                                 791
Card                                   1
Amount                           $150.06
Use Chip               Swipe Transaction
Merchant Name        8108107701014151249
Merchant City                      Burke
Merchant State                        VA
Zip                              22015.0
MCC                                 5300
Errors            [Insufficient Balance]
Is Fraud?                             No
Name: 9196386, dtype: object


#### "Is Fraud?"의 열 이름을 "Fraud"로 변경하고, True/False로 변경

In [13]:
transactions_df.rename(columns={'Is Fraud?': 'Fraud'}, inplace=True)

In [14]:
transactions_df['Fraud'] = transactions_df['Fraud'].map({'No': False, 'Yes': True})

#### "Amount"에서 $ 제거 및 float로 변환

In [15]:
transactions_df['Amount'] = transactions_df['Amount'].str.replace('$', '', regex=False).astype(float)


In [16]:
transactions_df

,Datetime,User,Card,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors,Fraud
9196264,1991-01-02 07:10:00,791,1,68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196265,1991-01-02 07:17:00,791,1,-68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196266,1991-01-02 07:21:00,791,1,113.62,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196267,1991-01-02 17:30:00,791,1,114.73,Swipe Transaction,-7269691894846892021,Burke,VA,22015.0,5411,NaN,False
9196268,1991-01-03 09:03:00,791,1,251.71,Swipe Transaction,-3693650930986299431,Burke,VA,22015.0,4814,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...
20357329,2020-02-28 23:51:00,1659,2,7.67,Chip Transaction,7231700044622779845,Cosby,TN,37722.0,5300,NaN,False
10296971,2020-02-28 23:53:00,863,1,49.06,Chip Transaction,7654254764356253071,North Brunswick,NJ,8902.0,5912,NaN,False
16828012,2020-02-28 23:56:00,1366,2,132.73,Chip Transaction,-7398558035733466800,Rockville Centre,NY,11570.0,5812,NaN,False
15999580,2020-02-28 23:56:00,1300,0,51.29,Online Transaction,-6458444334611773637,ONLINE,NaN,NaN,4784,NaN,False


In [17]:
transactions_df.dtypes

Datetime          datetime64[ns]
User                       int64
Card                       int64
Amount                   float64
Use Chip                  object
Merchant Name              int64
Merchant City             object
Merchant State            object
Zip                      float64
MCC                        int64
Errors                    object
Fraud                       bool
dtype: object

#### 각 거래에 UUID 할당

In [18]:
import uuid

transactions_df.insert(0, 'UUID', [str(uuid.uuid4()) for _ in range(len(transactions_df))])
transactions_df

,UUID,Datetime,User,Card,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors,Fraud
9196264,9a2a525b-f46b-4777-bfde-c68a1651c91a,1991-01-02 07:10:00,791,1,68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196265,2d2354f6-ccbf-44bc-bcc7-d78d3a7e9d19,1991-01-02 07:17:00,791,1,-68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196266,e13280ce-0079-4506-804b-a95d7d166471,1991-01-02 07:21:00,791,1,113.62,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196267,527a8019-6bba-4040-87bb-5e561a810b14,1991-01-02 17:30:00,791,1,114.73,Swipe Transaction,-7269691894846892021,Burke,VA,22015.0,5411,NaN,False
9196268,7e2d8063-7665-4719-a739-f1b04318839f,1991-01-03 09:03:00,791,1,251.71,Swipe Transaction,-3693650930986299431,Burke,VA,22015.0,4814,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20357329,7903ba77-3160-47af-9b75-7858aa2502ba,2020-02-28 23:51:00,1659,2,7.67,Chip Transaction,7231700044622779845,Cosby,TN,37722.0,5300,NaN,False
10296971,8fb6bb6e-6b63-4484-9fa9-4777e9555359,2020-02-28 23:53:00,863,1,49.06,Chip Transaction,7654254764356253071,North Brunswick,NJ,8902.0,5912,NaN,False
16828012,131a532d-ed3c-47b3-a79c-fb61892d64bf,2020-02-28 23:56:00,1366,2,132.73,Chip Transaction,-7398558035733466800,Rockville Centre,NY,11570.0,5812,NaN,False
15999580,71dd79b2-c182-4e79-9da0-4a3d72ca5569,2020-02-28 23:56:00,1300,0,51.29,Online Transaction,-6458444334611773637,ONLINE,NaN,NaN,4784,NaN,False


# Users data preprocessing

#### UUID 할당

In [19]:
import uuid

users_df.insert(0, 'UUID', [str(uuid.uuid4()) for _ in range(len(users_df))])
users_df

,UUID,Person,Current Age,Retirement Age,Birth Year,Birth Month,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,070a471d-9274-45b8-9a18-e3f28fe2bb9d,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5
1,36a31407-763c-475c-a93a-850b3cd010f6,Sasha Sadr,53,68,1966,12,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,$37891,$77254,$191349,701,5
2,80bca548-07e0-4305-903c-9155a2b79649,Saanvi Lee,81,67,1938,11,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,$22681,$33483,$196,698,5
3,bc03526a-4020-4755-8bb8-9c680108915e,Everlee Clark,63,63,1957,1,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,$163145,$249925,$202328,722,4
4,3897a2e8-1b03-408b-bc0f-eb4f5c32102b,Kyle Peterson,43,70,1976,9,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,$53797,$109687,$183855,675,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,71108ddf-29cf-49d1-97c9-13bd84c2ce4b,Jose Faraday,32,70,1987,7,Male,6577 Lexington Lane,9.0,Freeport,NY,11520,40.65,-73.58,$23550,$48010,$87837,703,3
1996,1151faa3-5880-4bbb-970a-1c93ce72f283,Ximena Richardson,62,65,1957,11,Female,2 Elm Drive,955.0,Independence,KY,41051,38.95,-84.54,$24218,$49378,$104480,740,4
1997,86661192-6b6c-4534-8e6a-7dde790a8f98,Annika Russell,47,67,1973,1,Female,276 Fifth Boulevard,NaN,Elizabeth,NJ,7201,40.66,-74.19,$15175,$30942,$71066,779,3
1998,0075e941-2bff-4c38-b659-7dba46e82e68,Juelz Roman,66,60,1954,2,Male,259 Valley Boulevard,NaN,Camp Hill,PA,17011,40.24,-76.92,$25336,$54654,$27241,618,1


#### "Birth Year" + "Birth Month" ==> "Birth"

In [20]:
users_df['Birth'] = pd.to_datetime(users_df['Birth Year'].astype(str) + '-' + users_df['Birth Month'].astype(str))
birth_idx = users_df.columns.get_loc('Birth Year')
birth_col = users_df.pop('Birth')
users_df.drop(columns=['Birth Year', 'Birth Month'], inplace=True)
users_df.insert(birth_idx, 'Birth', birth_col)

In [21]:
users_df.head()

,UUID,Person,Current Age,Retirement Age,Birth,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,070a471d-9274-45b8-9a18-e3f28fe2bb9d,Hazel Robinson,53,66,1966-11-01,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5
1,36a31407-763c-475c-a93a-850b3cd010f6,Sasha Sadr,53,68,1966-12-01,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,$37891,$77254,$191349,701,5
2,80bca548-07e0-4305-903c-9155a2b79649,Saanvi Lee,81,67,1938-11-01,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,$22681,$33483,$196,698,5
3,bc03526a-4020-4755-8bb8-9c680108915e,Everlee Clark,63,63,1957-01-01,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,$163145,$249925,$202328,722,4
4,3897a2e8-1b03-408b-bc0f-eb4f5c32102b,Kyle Peterson,43,70,1976-09-01,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,$53797,$109687,$183855,675,1


#### "Per Capita Income - Zipcode", "Yearly Income - Person", "Total Debt"에서 $ 제거 및 float변환

In [22]:
cols_to_fix = ["Per Capita Income - Zipcode", "Yearly Income - Person", "Total Debt"]
for col in cols_to_fix:
    users_df[col] = users_df[col].replace('[\$,]', '', regex=True).astype(float)

In [23]:
users_df.head()

,UUID,Person,Current Age,Retirement Age,Birth,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,070a471d-9274-45b8-9a18-e3f28fe2bb9d,Hazel Robinson,53,66,1966-11-01,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,29278.0,59696.0,127613.0,787,5
1,36a31407-763c-475c-a93a-850b3cd010f6,Sasha Sadr,53,68,1966-12-01,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,37891.0,77254.0,191349.0,701,5
2,80bca548-07e0-4305-903c-9155a2b79649,Saanvi Lee,81,67,1938-11-01,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,22681.0,33483.0,196.0,698,5
3,bc03526a-4020-4755-8bb8-9c680108915e,Everlee Clark,63,63,1957-01-01,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,163145.0,249925.0,202328.0,722,4
4,3897a2e8-1b03-408b-bc0f-eb4f5c32102b,Kyle Peterson,43,70,1976-09-01,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,53797.0,109687.0,183855.0,675,1


In [24]:
users_df.dtypes

UUID                                   object
Person                                 object
Current Age                             int64
Retirement Age                          int64
Birth                          datetime64[ns]
Gender                                 object
Address                                object
Apartment                             float64
City                                   object
State                                  object
Zipcode                                 int64
Latitude                              float64
Longitude                             float64
Per Capita Income - Zipcode           float64
Yearly Income - Person                float64
Total Debt                            float64
FICO Score                              int64
Num Credit Cards                        int64
dtype: object

# Cards data preprocessing

#### 각 카드별 UUID 할당

In [25]:
import uuid

cards_df.insert(0, 'UUID', [str(uuid.uuid4()) for _ in range(len(cards_df))])

#### "User"에 index대신 users_df의 UUID로 변환

In [26]:
cards_df['User'] = cards_df['User'].map(users_df['UUID'])

#### "Expires", "Acct Open Date"를 datetime으로 변경

In [27]:
from _codecs import encode
cards_df["Expires"] = pd.to_datetime(cards_df["Expires"], errors='coerce', format='mixed') + pd.offsets.MonthEnd(0)
cards_df["Acct Open Date"] = pd.to_datetime(cards_df["Acct Open Date"], errors='coerce', format='mixed')
cards_df["Expires"] = cards_df["Expires"].dt.normalize() + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
cards_df["Acct Open Date"] = cards_df["Acct Open Date"].dt.normalize()

#### "Credit Limit"에 $ 제거 및 float형 변환

In [28]:
cards_df['Credit Limit'] = cards_df['Credit Limit'].replace('[\$,]', '', regex=True).astype(float)

#### "Has Chip", "Card on Dark Web"을 bool type으로 변환

In [29]:
cards_df['Has Chip'] = cards_df['Has Chip'].map({'NO': False, 'YES': True})
cards_df["Card on Dark Web"] = cards_df["Card on Dark Web"].map({'No': False, 'Yes': True})

In [30]:
cards_df

,UUID,User,CARD INDEX,Card Brand,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,0421b4c3-16f4-4cfa-adfd-21f54051776d,070a471d-9274-45b8-9a18-e3f28fe2bb9d,0,Visa,Debit,4344676511950444,2022-12-31 23:59:59,623,True,2,24295.0,2002-09-01,2008,False
1,8e0f1ad1-37fd-4ede-ba7c-ec1b1a62a267,070a471d-9274-45b8-9a18-e3f28fe2bb9d,1,Visa,Debit,4956965974959986,2020-12-31 23:59:59,393,True,2,21968.0,2014-04-01,2014,False
2,246275f6-9db6-409a-a066-747ca9f1ead6,070a471d-9274-45b8-9a18-e3f28fe2bb9d,2,Visa,Debit,4582313478255491,2024-02-29 23:59:59,719,True,2,46414.0,2003-07-01,2004,False
3,3ab9debd-dd09-4d5d-acdb-8e76c6fb94c3,070a471d-9274-45b8-9a18-e3f28fe2bb9d,3,Visa,Credit,4879494103069057,2024-08-31 23:59:59,693,False,1,12400.0,2003-01-01,2012,False
4,699fcacc-4011-48ca-8bc6-9d4f254301ec,070a471d-9274-45b8-9a18-e3f28fe2bb9d,4,Mastercard,Debit (Prepaid),5722874738736011,2009-03-31 23:59:59,75,True,1,28.0,2008-09-01,2009,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6141,9cfeb1ec-a77c-45e8-996c-effa0cfc71ca,86661192-6b6c-4534-8e6a-7dde790a8f98,1,Amex,Credit,300609782832003,2024-01-31 23:59:59,663,True,1,6900.0,2000-11-01,2013,False
6142,014796b6-a369-4a07-ba1f-e55f7c48d7e1,86661192-6b6c-4534-8e6a-7dde790a8f98,2,Visa,Credit,4718517475996018,2021-01-31 23:59:59,492,True,2,5700.0,2012-04-01,2012,False
6143,419ac96f-689b-4081-aa3c-33e87577265d,0075e941-2bff-4c38-b659-7dba46e82e68,0,Mastercard,Credit,5929512204765914,2020-08-31 23:59:59,237,False,2,9200.0,2012-02-01,2012,False
6144,1b55d0b3-c5bb-4640-a6bf-0355b1c1e338,555aec74-8c49-473c-a54a-9e4dd4d79a7e,0,Mastercard,Debit,5589768928167462,2020-01-31 23:59:59,630,True,1,28074.0,2020-01-01,2020,False


In [31]:
cards_df.dtypes

UUID                             object
User                             object
CARD INDEX                        int64
Card Brand                       object
Card Type                        object
Card Number                       int64
Expires                  datetime64[ns]
CVV                               int64
Has Chip                           bool
Cards Issued                      int64
Credit Limit                    float64
Acct Open Date           datetime64[ns]
Year PIN last Changed             int64
Card on Dark Web                   bool
dtype: object

# Users data preprocessing - 2

#### users_df에 각 user의 card의 UUID 추가

In [32]:
users_df['Cards'] = users_df['UUID'].map(cards_df.groupby('User')['UUID'].apply(list)).apply(lambda x: x if isinstance(x, list) else [])

In [33]:
users_df.head()

,UUID,Person,Current Age,Retirement Age,Birth,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards,Cards
0,070a471d-9274-45b8-9a18-e3f28fe2bb9d,Hazel Robinson,53,66,1966-11-01,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,29278.0,59696.0,127613.0,787,5,"[0421b4c3-16f4-4cfa-adfd-21f54051776d, 8e0f1ad..."
1,36a31407-763c-475c-a93a-850b3cd010f6,Sasha Sadr,53,68,1966-12-01,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,37891.0,77254.0,191349.0,701,5,"[3b220800-637f-4376-8b39-be8896d155ad, 710852a..."
2,80bca548-07e0-4305-903c-9155a2b79649,Saanvi Lee,81,67,1938-11-01,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,22681.0,33483.0,196.0,698,5,"[90b5f915-ba01-43ad-89cb-08bdaca45f2d, 101fe56..."
3,bc03526a-4020-4755-8bb8-9c680108915e,Everlee Clark,63,63,1957-01-01,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,163145.0,249925.0,202328.0,722,4,"[e0d9610b-d2b8-4081-b154-a6b66165ad15, 6ad825e..."
4,3897a2e8-1b03-408b-bc0f-eb4f5c32102b,Kyle Peterson,43,70,1976-09-01,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,53797.0,109687.0,183855.0,675,1,[87471b18-052a-487d-95fc-9eb138d9d948]


# Transactions data preprocessing - 2

#### "User", "Card" 열을 각각 users_df의 UUID, cards_df의 UUID로 변경

In [34]:
transactions_df['User'] = transactions_df['User'].map(users_df['UUID'])

In [35]:
transactions_df = transactions_df.merge(
    cards_df[['User', 'CARD INDEX', 'UUID']].rename(columns={'UUID': 'UUID_card'}), 
    left_on=['User', 'Card'], 
    right_on=['User', 'CARD INDEX'], 
    how='left'
)
transactions_df['Card'] = transactions_df['UUID_card']
transactions_df.drop(columns=['CARD INDEX', 'UUID_card'], inplace=True)

#### merchants_df 분리

In [36]:
merchants_df = transactions_df[['Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC']].copy()

In [37]:
import uuid

merchants_df = merchants_df.drop_duplicates().reset_index(drop=True)
merchants_df.insert(0, 'UUID', [str(uuid.uuid4()) for _ in range(len(merchants_df))])

merchants_df

,UUID,Merchant Name,Merchant City,Merchant State,Zip,MCC
0,f00a4ef9-043d-438f-9ea5-8db9f85967b6,2027553650310142703,Burke,VA,22015.0,5541
1,a22479a7-b4e1-41a9-b057-78f30bb10cae,-7269691894846892021,Burke,VA,22015.0,5411
2,4e966077-7a47-4fc8-a48f-2711bc4c6d39,-3693650930986299431,Burke,VA,22015.0,4814
3,570d9eaf-a25e-449e-bd76-7ea59d2b40fe,3189517333335617109,Fairfax,VA,22030.0,5311
4,8d486481-b3cf-43cf-a279-591bf2c5c99b,5701841789931834110,Burke,VA,22015.0,5411
...,...,...,...,...,...,...
283976,b41c0a59-1c1e-416c-9169-75d5a1216d70,-7911106634782211204,Portland,ME,4103.0,7349
283977,6d2489eb-e95f-40da-9945-4de1c478c351,-8893969934710155907,Perris,CA,92570.0,7538
283978,a0816a4d-b76e-4aff-8343-35329a33e347,-2248444526575079323,Phoenix,AZ,85043.0,5655
283979,aff3d6ae-ebcd-49e1-9cb0-a2a5bc7ba10e,-7664872663026589029,Bronx,NY,10455.0,8021


#### transactions_df의 "Merchant Name" 자리에 merchants_df의 "UUID"로 삽입

In [38]:
# transactions_df와 병합
transactions_df = transactions_df.merge(
    merchants_df.rename(columns={'UUID': 'Merchant'}), 
    on=['Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC'], 
    how='left'
)

# 'Merchant' 컬럼을 'Merchant Name' 위치로 이동
cols = transactions_df.columns.tolist()
merchant_name_idx = cols.index('Merchant Name')
cols.insert(merchant_name_idx, cols.pop(cols.index('Merchant')))
transactions_df = transactions_df[cols]

#### transactions_df에서 "Merchant"만 놔두고, "Merchant Name", "Merchant City", "Merchant State", "Zip", "MCC"는 drop

In [39]:
transactions_df.drop(columns=["Merchant Name", "Merchant City", "Merchant State", "Zip", "MCC"], inplace=True)

In [40]:
transactions_df

,UUID,Datetime,User,Card,Amount,Use Chip,Merchant,Errors,Fraud
0,9a2a525b-f46b-4777-bfde-c68a1651c91a,1991-01-02 07:10:00,204839a6-a157-47f2-9cce-7e25b1e3b056,b14884b6-3df1-4281-8be8-f2e44b252197,68.00,Swipe Transaction,f00a4ef9-043d-438f-9ea5-8db9f85967b6,NaN,False
1,2d2354f6-ccbf-44bc-bcc7-d78d3a7e9d19,1991-01-02 07:17:00,204839a6-a157-47f2-9cce-7e25b1e3b056,b14884b6-3df1-4281-8be8-f2e44b252197,-68.00,Swipe Transaction,f00a4ef9-043d-438f-9ea5-8db9f85967b6,NaN,False
2,e13280ce-0079-4506-804b-a95d7d166471,1991-01-02 07:21:00,204839a6-a157-47f2-9cce-7e25b1e3b056,b14884b6-3df1-4281-8be8-f2e44b252197,113.62,Swipe Transaction,f00a4ef9-043d-438f-9ea5-8db9f85967b6,NaN,False
3,527a8019-6bba-4040-87bb-5e561a810b14,1991-01-02 17:30:00,204839a6-a157-47f2-9cce-7e25b1e3b056,b14884b6-3df1-4281-8be8-f2e44b252197,114.73,Swipe Transaction,a22479a7-b4e1-41a9-b057-78f30bb10cae,NaN,False
4,7e2d8063-7665-4719-a739-f1b04318839f,1991-01-03 09:03:00,204839a6-a157-47f2-9cce-7e25b1e3b056,b14884b6-3df1-4281-8be8-f2e44b252197,251.71,Swipe Transaction,4e966077-7a47-4fc8-a48f-2711bc4c6d39,NaN,False
...,...,...,...,...,...,...,...,...,...
24386895,7903ba77-3160-47af-9b75-7858aa2502ba,2020-02-28 23:51:00,5c2e7a7f-6296-456d-b992-2e859ac7f945,64c76157-eb7b-4590-982f-a3bfb74db05b,7.67,Chip Transaction,f02db878-741d-47ab-9f68-d225c3a4e9bd,NaN,False
24386896,8fb6bb6e-6b63-4484-9fa9-4777e9555359,2020-02-28 23:53:00,66e2593e-243c-4746-88bd-604c6a50930a,44a18adb-0c3e-4cda-b131-e2cf5247e68b,49.06,Chip Transaction,a444462b-7e27-4646-b33a-7222b0ef44e8,NaN,False
24386897,131a532d-ed3c-47b3-a79c-fb61892d64bf,2020-02-28 23:56:00,348acf8b-f35c-4aad-af2d-0c0a0f702ec5,5853389e-903c-4c34-90f8-f3ac61b84a2b,132.73,Chip Transaction,0fd5dfdd-de93-4280-b8d9-2d0a536f5752,NaN,False
24386898,71dd79b2-c182-4e79-9da0-4a3d72ca5569,2020-02-28 23:56:00,761b0cb6-050d-47f2-b3c5-620871195de4,6e57614f-74e1-4352-b6a0-96538e094585,51.29,Online Transaction,ff62f1f5-418d-4fd5-a855-2ef810fef290,NaN,False


# Cards data preprocessing - 2

#### 카드 개설일(Acct Open Date)보다 거래일이 빠른 카드는 거래일을 기준으로 개설일 수정

In [41]:
# 거래 시점이 카드 개설일보다 빠른 거래 찾기
open_date_map = cards_df.set_index("UUID")["Acct Open Date"]
mask = transactions_df["Datetime"] < transactions_df["Card"].map(open_date_map)

# 카드별로 가장 이른 거래일(00:00:00)로 보정값 생성
fix_open_date_by_card = (
    transactions_df.loc[mask]
    .groupby("Card")["Datetime"]
    .min()
    .dt.normalize()
)

# cards_df의 Acct Open Date 수정
cards_df.loc[cards_df["UUID"].isin(fix_open_date_by_card.index), "Acct Open Date"] = (
    cards_df.loc[cards_df["UUID"].isin(fix_open_date_by_card.index), "UUID"]
    .map(fix_open_date_by_card)
)

print(f"수정된 카드 수: {fix_open_date_by_card.shape[0]:,}")
print(f"조건에 해당하는 거래 수: {mask.sum():,}")

수정된 카드 수: 1,167
조건에 해당하는 거래 수: 1,496


# Merchants data preprocessing

#### column명 수정

In [42]:
merchants_df = merchants_df.rename(columns={'Merchant Name': 'Name', 'Merchant City': 'City', 'Merchant State': 'State', 'Zip':'Zipcode'})

In [43]:
merchants_df['Zipcode'] = pd.to_numeric(merchants_df['Zipcode'], errors='coerce').astype('Int64')

In [44]:
merchants_df

,UUID,Name,City,State,Zipcode,MCC
0,f00a4ef9-043d-438f-9ea5-8db9f85967b6,2027553650310142703,Burke,VA,22015,5541
1,a22479a7-b4e1-41a9-b057-78f30bb10cae,-7269691894846892021,Burke,VA,22015,5411
2,4e966077-7a47-4fc8-a48f-2711bc4c6d39,-3693650930986299431,Burke,VA,22015,4814
3,570d9eaf-a25e-449e-bd76-7ea59d2b40fe,3189517333335617109,Fairfax,VA,22030,5311
4,8d486481-b3cf-43cf-a279-591bf2c5c99b,5701841789931834110,Burke,VA,22015,5411
...,...,...,...,...,...,...
283976,b41c0a59-1c1e-416c-9169-75d5a1216d70,-7911106634782211204,Portland,ME,4103,7349
283977,6d2489eb-e95f-40da-9945-4de1c478c351,-8893969934710155907,Perris,CA,92570,7538
283978,a0816a4d-b76e-4aff-8343-35329a33e347,-2248444526575079323,Phoenix,AZ,85043,5655
283979,aff3d6ae-ebcd-49e1-9cb0-a2a5bc7ba10e,-7664872663026589029,Bronx,NY,10455,8021


In [45]:
merchants_df.dtypes

UUID       object
Name        int64
City       object
State      object
Zipcode     Int64
MCC         int64
dtype: object

In [46]:
users_df.head()

,UUID,Person,Current Age,Retirement Age,Birth,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards,Cards
0,070a471d-9274-45b8-9a18-e3f28fe2bb9d,Hazel Robinson,53,66,1966-11-01,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,29278.0,59696.0,127613.0,787,5,"[0421b4c3-16f4-4cfa-adfd-21f54051776d, 8e0f1ad..."
1,36a31407-763c-475c-a93a-850b3cd010f6,Sasha Sadr,53,68,1966-12-01,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,37891.0,77254.0,191349.0,701,5,"[3b220800-637f-4376-8b39-be8896d155ad, 710852a..."
2,80bca548-07e0-4305-903c-9155a2b79649,Saanvi Lee,81,67,1938-11-01,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,22681.0,33483.0,196.0,698,5,"[90b5f915-ba01-43ad-89cb-08bdaca45f2d, 101fe56..."
3,bc03526a-4020-4755-8bb8-9c680108915e,Everlee Clark,63,63,1957-01-01,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,163145.0,249925.0,202328.0,722,4,"[e0d9610b-d2b8-4081-b154-a6b66165ad15, 6ad825e..."
4,3897a2e8-1b03-408b-bc0f-eb4f5c32102b,Kyle Peterson,43,70,1976-09-01,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,53797.0,109687.0,183855.0,675,1,[87471b18-052a-487d-95fc-9eb138d9d948]


In [47]:
cards_df.head()

,UUID,User,CARD INDEX,Card Brand,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,0421b4c3-16f4-4cfa-adfd-21f54051776d,070a471d-9274-45b8-9a18-e3f28fe2bb9d,0,Visa,Debit,4344676511950444,2022-12-31 23:59:59,623,True,2,24295.0,2002-09-01,2008,False
1,8e0f1ad1-37fd-4ede-ba7c-ec1b1a62a267,070a471d-9274-45b8-9a18-e3f28fe2bb9d,1,Visa,Debit,4956965974959986,2020-12-31 23:59:59,393,True,2,21968.0,2014-04-01,2014,False
2,246275f6-9db6-409a-a066-747ca9f1ead6,070a471d-9274-45b8-9a18-e3f28fe2bb9d,2,Visa,Debit,4582313478255491,2024-02-29 23:59:59,719,True,2,46414.0,2003-07-01,2004,False
3,3ab9debd-dd09-4d5d-acdb-8e76c6fb94c3,070a471d-9274-45b8-9a18-e3f28fe2bb9d,3,Visa,Credit,4879494103069057,2024-08-31 23:59:59,693,False,1,12400.0,2003-01-01,2012,False
4,699fcacc-4011-48ca-8bc6-9d4f254301ec,070a471d-9274-45b8-9a18-e3f28fe2bb9d,4,Mastercard,Debit (Prepaid),5722874738736011,2009-03-31 23:59:59,75,True,1,28.0,2008-09-01,2009,False


# Save processed data

In [48]:
import os

result_dir = "data/processed_data"
os.makedirs(result_dir, exist_ok=True)

transactions_df.to_csv(os.path.join(result_dir, "transactions.csv"), index=False)
cards_df.to_csv(os.path.join(result_dir, "cards.csv"), index=False)
users_df.to_csv(os.path.join(result_dir, "users.csv"), index=False)
merchants_df.to_csv(os.path.join(result_dir, "merchants.csv"), index=False)